# Why Training Generalizes: Model Complexity & the Generalization Bound

**A Pokemon vs. Digimon story.** Suppose we want a tiny model that decides whether a creature drawing is a *Pokemon* or a *Digimon*. We boil each image down to one number - how *busy* its outline is - and classify with a single threshold. The model could not be simpler, yet it lets us ask the deepest question in machine learning:

> Why should a low error on the images we trained on tell us anything about images we have never seen?

This notebook builds the answer end to end. Roadmap:

1. **The feature** - turn an image into one number e (edge-pixel count).
2. **The model** - a threshold classifier f_h and the hypothesis set H (model complexity |H|).
3. **The loss** - the 0-1 error L(h, D) and the best threshold.
4. **The gap** - why optimizing on a finite sample D_train can mislead us about D_all.
5. **The bound** - union bound + Hoeffding: P(bad) <= |H| * 2 * exp(-2 * N * eps^2).
6. **The tradeoff** - bigger |H| lowers achievable loss but widens the gap.

Run the cells top to bottom; every section ends in an interactive widget you can play with.

In [ ]:
# C02 - Environment setup, shared imports, RNG, and global constants
%matplotlib inline
import sys, subprocess
import numpy as np
import matplotlib.pyplot as plt
from scipy import ndimage

# ipywidgets is optional: install on a fresh Colab runtime, else degrade gracefully.
HAS_WIDGETS = True
try:
    import ipywidgets as widgets
    from ipywidgets import interact, IntSlider, FloatSlider
except Exception:
    try:
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'ipywidgets'], check=False)
        import ipywidgets as widgets
        from ipywidgets import interact, IntSlider, FloatSlider
    except Exception:
        HAS_WIDGETS = False
        def interact(_fn=None, **kwargs):
            return None

# Reproducibility
SEED = 7
rng = np.random.default_rng(SEED)

# Global constants: the feature range and the hypothesis grid H = {1, ..., 10000}
E_MAX = 10000
H_GRID = np.arange(1, E_MAX + 1)          # |H| = 10000 candidate thresholds

def show_interactive(render_fn, make_widgets, defaults):
    # Use a live widget when ipywidgets is present, else render once with defaults.
    if HAS_WIDGETS:
        return interact(render_fn, **make_widgets())
    print('[ipywidgets unavailable - static render with default values]')
    return render_fn(**defaults)

print(f'NumPy {np.__version__} | widgets available: {HAS_WIDGETS}')
print(f'E_MAX = {E_MAX} | |H| = {len(H_GRID)} candidate thresholds')

## Section 1 - From a picture to a single number

Digimon tend to have **busier outlines** than Pokemon: more spikes, ridges, and internal detail. If we run an edge detector (convert to grayscale, then take the image gradient with a Sobel/Canny-style filter) and *count the edge pixels*, busy creatures light up far more pixels than smooth ones.

That single count - call it **e** - is the only feature we will use. On the lecture slides two example creatures gave **e = 3558** (a simple Pokemon) versus **e = 7389** (a busy Digimon). The next cell reproduces that contrast on two synthetic images so the idea is concrete and runs with no external dataset.

In [ ]:
# C04 - Make the edge-count feature tangible on two synthetic grayscale images
def edge_count(img, thresh_frac=0.30):
    # Sobel gradient-magnitude edge detector. Returns (count, edge_mask, magnitude).
    gx = ndimage.sobel(img.astype(float), axis=0, mode='reflect')
    gy = ndimage.sobel(img.astype(float), axis=1, mode='reflect')
    mag = np.hypot(gx, gy)
    mmax = mag.max() if mag.max() > 0 else 1.0
    mask = mag > (thresh_frac * mmax)
    return int(mask.sum()), mask, mag

def make_simple_image(size=128):
    # A Pokemon-like simple round blob: one smooth, clean outline.
    yy, xx = np.mgrid[0:size, 0:size]
    cx = cy = size // 2
    r = np.hypot(xx - cx, yy - cy)
    img = (r < size * 0.32).astype(float)
    return ndimage.gaussian_filter(img, sigma=1.5)

def make_busy_image(size=128, seed=0):
    # A Digimon-like busy shape: blob + radial spikes + spots + interior texture.
    local = np.random.default_rng(seed)
    yy, xx = np.mgrid[0:size, 0:size]
    cx = cy = size // 2
    r = np.hypot(xx - cx, yy - cy)
    ang = np.arctan2(yy - cy, xx - cx)
    img = (r < size * 0.32).astype(float)
    spikes = (np.abs(np.sin(ang * 9)) > 0.85) & (r < size * 0.45)
    img[spikes] = 1.0
    for _ in range(40):
        sx = int(local.integers(8, size - 8)); sy = int(local.integers(8, size - 8))
        rad = int(local.integers(2, 6))
        img[np.hypot(xx - sx, yy - sy) < rad] = local.uniform(0.0, 1.0)
    body = r < size * 0.32
    noise = local.standard_normal((size, size)) * 0.4
    img[body] = np.clip(img[body] + noise[body], 0, 1)
    return ndimage.gaussian_filter(img, sigma=0.6)

simple_img = make_simple_image()
busy_img = make_busy_image(seed=12345)

def render_edges(thresh_frac=0.30):
    e_s, mask_s, _ = edge_count(simple_img, thresh_frac)
    e_b, mask_b, _ = edge_count(busy_img, thresh_frac)
    fig, ax = plt.subplots(2, 2, figsize=(8, 8))
    ax[0, 0].imshow(simple_img, cmap='gray'); ax[0, 0].set_title('Pokemon-like (simple)')
    ax[0, 1].imshow(mask_s, cmap='gray'); ax[0, 1].set_title(f'edges: e = {e_s}')
    ax[1, 0].imshow(busy_img, cmap='gray'); ax[1, 0].set_title('Digimon-like (busy)')
    ax[1, 1].imshow(mask_b, cmap='gray'); ax[1, 1].set_title(f'edges: e = {e_b}')
    for a in ax.ravel():
        a.axis('off')
    fig.suptitle(f'Edge count e at threshold fraction = {thresh_frac:.2f}')
    plt.tight_layout(); plt.show()

# Reference edge counts at the default threshold (simple < busy, mirroring the slides)
example_simple_e, _, _ = edge_count(simple_img)
example_busy_e, _, _ = edge_count(busy_img)
print(f'e(simple) = {example_simple_e}   e(busy) = {example_busy_e}')

show_interactive(
    render_edges,
    lambda: {'thresh_frac': FloatSlider(value=0.30, min=0.05, max=0.80, step=0.05, description='thresh')},
    {'thresh_frac': 0.30},
)

## From one image to a whole population

To talk about *generalization* we need many images, not two. The original Kaggle/Qiita Pokemon and Digimon image sets are not bundled with this notebook, so to stay fully offline and deterministic we **simulate** each class's edge-count distribution from the shapes shown on the slides (roughly 819 Pokemon and 971 Digimon, with **overlapping** e distributions).

The overlap is the whole point: because some Pokemon are busy and some Digimon are smooth, no single threshold can separate them perfectly. The next cell builds this fixed population D_all - the ground truth we will sample from for the rest of the notebook.

In [ ]:
# C06 - Build the ground-truth population D_all by simulating slide edge-count histograms
N_pokemon, N_digimon = 819, 971                 # slide image counts (p.14-16)

def _clipped_normal(n, mu, sigma, lo=0, hi=E_MAX, gen=rng):
    return np.clip(gen.normal(mu, sigma, size=n), lo, hi)

# Overlapping distributions calibrated so h_all ~ 4824 and L(h_all, D_all) ~ 0.28
e_pokemon = _clipped_normal(N_pokemon, mu=4200, sigma=2800)
e_digimon = _clipped_normal(N_digimon, mu=6900, sigma=2800)

e_all = np.concatenate([e_pokemon, e_digimon])
y_all = np.concatenate([np.zeros(N_pokemon, dtype=int), np.ones(N_digimon, dtype=int)])
perm = rng.permutation(e_all.size)              # shuffle so order carries no label info
e_all, y_all = e_all[perm], y_all[perm]
N_all = int(e_all.size)
D_all = {'e': e_all, 'y': y_all, 'N': N_all}

fig, ax = plt.subplots(figsize=(8, 4.5))
bins = np.linspace(0, E_MAX, 60)
ax.hist(e_all[y_all == 0], bins=bins, alpha=0.6, label='Pokemon (y=0)', color='#1f77b4')
ax.hist(e_all[y_all == 1], bins=bins, alpha=0.6, label='Digimon (y=1)', color='#d62728')
ax.set_xlabel('edge-pixel count  e'); ax.set_ylabel('# images')
ax.set_title(f'D_all: {N_all} images - overlapping edge-count distributions')
ax.legend(); plt.tight_layout(); plt.show()
print(f'N_all = {N_all}  (Pokemon {N_pokemon}, Digimon {N_digimon})')

## Section 2 - The model: a threshold classifier and its hypothesis set

**Step 1 of machine learning: pick a function with an unknown parameter.** Our function is a threshold rule:

    f_h(e) = 1 (Digimon)  if e >= h
    f_h(e) = 0 (Pokemon)  if e <  h

The unknown parameter is the threshold h. Allowing every integer threshold from 1 to 10000 gives the **hypothesis set** H = {1, ..., 10000}, so its size is **|H| = 10000**. This number is the model's *complexity*, and it is the single quantity the entire generalization argument will hinge on.

In [ ]:
# C08 - The threshold classifier f_h(e) = 1[e >= h] and an interactive decision boundary
def f_h(e, h):
    # Predict 1 (Digimon) when edge count e >= threshold h, else 0 (Pokemon).
    return (np.asarray(e) >= h).astype(int)

def predict(e, h):
    return f_h(e, h)

print(f'Model complexity: |H| = {len(H_GRID)} candidate thresholds')

def render_boundary(h=4824):
    preds = f_h(e_all, h)
    acc = float(np.mean(preds == y_all))
    fig, ax = plt.subplots(figsize=(8, 4.5))
    bins = np.linspace(0, E_MAX, 60)
    ax.hist(e_all[y_all == 0], bins=bins, alpha=0.55, label='Pokemon (y=0)', color='#1f77b4')
    ax.hist(e_all[y_all == 1], bins=bins, alpha=0.55, label='Digimon (y=1)', color='#d62728')
    ax.axvline(h, color='black', lw=2, ls='--', label=f'threshold h = {h}')
    ax.set_xlabel('edge-pixel count  e'); ax.set_ylabel('# images')
    ax.set_title(f'Predict Digimon to the right of h   (accuracy on D_all = {acc:.3f})')
    ax.legend(); plt.tight_layout(); plt.show()

show_interactive(
    render_boundary,
    lambda: {'h': IntSlider(value=4824, min=int(H_GRID.min()), max=int(H_GRID.max()), step=100, description='h')},
    {'h': 4824},
)

## Section 3 - The loss: measuring how good a threshold is

**Step 2 of machine learning: define a loss.** For a classifier we use the empirical **0-1 loss** on a dataset D of N examples:

    L(h, D) = (1/N) * sum over examples of 1[ f_h(e) != y ]

i.e. the fraction of images the threshold gets wrong. Lower is better. (Cross-entropy is a common smooth alternative when we need gradients, but for a hard threshold the 0-1 error is exactly what we care about.) The next cell sweeps L over every threshold in H and finds the loss-minimizing one.

In [ ]:
# C10 - The empirical 0-1 loss, the full loss-vs-threshold sweep, and the ideal threshold
def loss_L(h, e, y):
    # 0-1 loss of f_h. h may be a scalar or an array of thresholds (vectorized).
    h_arr = np.atleast_1d(np.asarray(h))
    pred = (e[None, :] >= h_arr[:, None]).astype(np.int8)      # (len(h), N)
    err = np.mean(pred != y[None, :], axis=1)                  # (len(h),)
    return err if np.ndim(h) else float(err[0])

loss_curve_all = loss_L(H_GRID, e_all, y_all)
best_idx = int(np.argmin(loss_curve_all))
h_all = int(H_GRID[best_idx])
L_all = float(loss_curve_all[best_idx])

def render_loss(h=4824):
    L_here = loss_L(h, e_all, y_all)
    fig, ax = plt.subplots(figsize=(8, 4.5))
    ax.plot(H_GRID, loss_curve_all, color='#2ca02c', lw=1.5, label='L(h, D_all)')
    ax.scatter([h_all], [L_all], color='black', zorder=5, label=f'ideal h_all = {h_all}, L = {L_all:.3f}')
    ax.axvline(h, color='gray', ls='--', label=f'h = {h}: L = {L_here:.3f}')
    ax.set_xlabel('threshold h'); ax.set_ylabel('0-1 loss L(h, D_all)')
    ax.set_title('Loss vs threshold on the full population D_all')
    ax.legend(); plt.tight_layout(); plt.show()

print(f'Ideal threshold h_all = {h_all}   with loss L(h_all, D_all) = {L_all:.3f}')
show_interactive(
    render_loss,
    lambda: {'h': IntSlider(value=h_all, min=int(H_GRID.min()), max=int(H_GRID.max()), step=100, description='h')},
    {'h': h_all},
)

## Section 4 - The catch: we can only optimize on a sample

In reality we never see the full population D_all. We see a finite, i.i.d. **training set** D_train, and we pick the threshold that looks best *there*:

    h_train = argmin_h  L(h, D_train)

But what we actually care about is the loss on everything, L(h_train, D_all). These can disagree badly: a worked slide example finds a threshold with **0.20** loss on the training set yet **0.37** on the full population. That difference - the **generalization gap** - is what the next cell makes visible by sampling D_train and comparing both loss curves.

In [ ]:
# C12 - The generalization gap: optimize on D_train, evaluate on D_all
def _fit_on_sample(N, seed):
    gen = np.random.default_rng(seed)
    idx = gen.integers(0, N_all, size=N)            # i.i.d. draws (with replacement) from D_all
    e_tr, y_tr = e_all[idx], y_all[idx]
    curve_tr = loss_L(H_GRID, e_tr, y_tr)
    h_tr = int(H_GRID[int(np.argmin(curve_tr))])
    return e_tr, y_tr, h_tr, curve_tr

# Default sample fixes the required output variables before any widget interaction.
_e_tr, _y_tr, h_train, _curve_tr = _fit_on_sample(N=300, seed=0)
sample_D_train = {'e': _e_tr, 'y': _y_tr, 'N': 300}

def render_gap(N=300, seed=0):
    e_tr, y_tr, h_tr, curve_tr = _fit_on_sample(N, seed)
    L_tr_tr = loss_L(h_tr, e_tr, y_tr)
    L_tr_all = loss_L(h_tr, e_all, y_all)
    gap = L_tr_all - L_tr_tr
    fig, ax = plt.subplots(figsize=(8, 4.5))
    ax.plot(H_GRID, loss_curve_all, color='#2ca02c', lw=1.5, label='L(h, D_all)')
    ax.plot(H_GRID, curve_tr, color='#9467bd', lw=1.2, alpha=0.8, label=f'L(h, D_train) N={N}')
    ax.axvline(h_all, color='black', ls=':', label=f'h_all = {h_all}')
    ax.axvline(h_tr, color='#9467bd', ls='--', label=f'h_train = {h_tr}')
    ax.set_xlabel('threshold h'); ax.set_ylabel('0-1 loss')
    ax.set_title('Train vs population loss - the generalization gap')
    ax.legend(); plt.tight_layout(); plt.show()
    print(f'L(h_train,D_train) = {L_tr_tr:.3f}   L(h_train,D_all) = {L_tr_all:.3f}   gap = {gap:+.3f}   ideal L_all = {L_all:.3f}')

gap_demo = render_gap
show_interactive(
    render_gap,
    lambda: {'N': IntSlider(value=300, min=50, max=1000, step=50, description='N'),
             'seed': IntSlider(value=0, min=0, max=50, step=1, description='seed')},
    {'N': 300, 'seed': 0},
)

## Section 5 - Bounding how often training misleads us

When does a training set fool us? Call D_train **bad** if *some* hypothesis has a large train-vs-population gap:

    bad  <=>  exists h in H with  |L(h, D_train) - L(h, D_all)| > eps

Two classic tools bound the chance of this:

- **Hoeffding's inequality** (per hypothesis): for a single fixed h, P(|L(h,D_train) - L(h,D_all)| > eps) <= 2 * exp(-2 * N * eps^2).
- **Union bound** (over the whole set): the probability that *any* of the |H| hypotheses is off is at most the sum of the individual probabilities.

Putting them together:

    P(D_train is bad) <= |H| * 2 * exp(-2 * N * eps^2)

So reliability improves with more data N and degrades with more complexity |H|. The next cell checks this bound by Monte-Carlo.

In [ ]:
# C14 - Monte-Carlo check of the bound  P(bad) <= |H| * 2 * exp(-2 * N * eps^2)
H_COARSE = H_GRID[::100]                       # coarse threshold subset keeps the simulation responsive
loss_all_coarse = loss_L(H_COARSE, e_all, y_all)

def estimate_p_bad(N, eps, trials, seed=1):
    gen = np.random.default_rng(seed)
    idx = gen.integers(0, N_all, size=(trials, N))
    e_tr = e_all[idx]; y_tr = y_all[idx]                                  # (trials, N)
    pred = (e_tr[:, None, :] >= H_COARSE[None, :, None]).astype(np.int8)  # (trials, Hc, N)
    loss_tr = (pred != y_tr[:, None, :]).mean(axis=2)                     # (trials, Hc)
    max_dev = np.max(np.abs(loss_tr - loss_all_coarse[None, :]), axis=1)  # (trials,)
    return float(np.mean(max_dev > eps))

def render_bound(eps=0.10, trials=150):
    N_values = np.array([50, 100, 200, 300, 500, 800])
    emp = np.array([estimate_p_bad(int(N), eps, trials, seed=1) for N in N_values])
    bound = np.minimum(1.0, len(H_GRID) * 2 * np.exp(-2 * N_values * eps**2))
    fig, ax = plt.subplots(figsize=(8, 4.5))
    ax.plot(N_values, bound, 'o-', color='#ff7f0e', label='Hoeffding/union bound')
    ax.plot(N_values, emp, 's--', color='#1f77b4', label='empirical P(bad) [coarse H]')
    ax.set_xlabel('training-set size N'); ax.set_ylabel(f'P(bad)  (eps = {eps:.2f})')
    ax.set_title('Empirical badness vs theoretical upper bound')
    ax.set_ylim(-0.03, 1.05); ax.legend(); plt.tight_layout()
    render_bound.fig = fig
    render_bound.emp = emp
    plt.show()
    print('bound is a valid upper envelope:', bool(np.all(emp <= bound + 1e-9)),
          '(coarse H under-counts the true worst-case deviation)')

render_bound(eps=0.10, trials=150)
monte_carlo_bad_prob = render_bound.emp
bound_compare_fig = render_bound.fig
if HAS_WIDGETS:
    interact(render_bound,
             eps=FloatSlider(value=0.10, min=0.03, max=0.30, step=0.01, description='eps'),
             trials=IntSlider(value=150, min=50, max=400, step=50, description='trials'))

## Section 6 - The payoff: the model-complexity tradeoff

The bound exposes a tension in choosing |H|:

- **Bigger |H|** (more candidate thresholds / a richer model) can only *lower* the best achievable loss min_h L(h, D_all) - more functions to choose from.
- **But bigger |H|** also *widens* the guaranteed gap, because the bound grows with |H|.
- **Smaller |H|** does the reverse: a safer gap, but a worse floor on achievable loss.

The good choice minimizes their **sum**. This is the bias/complexity tradeoff - and it is exactly why we eventually want models (like deep networks) that can push the achievable loss down *while* keeping the effective complexity in check. The next cell traces both curves and marks the sweet spot.

In [ ]:
# C16 - The model-complexity tradeoff: achievable loss vs generalization-gap bound
H_SIZES = np.array([2, 5, 10, 20, 50, 100, 200, 500, 1000, 2000, 5000, 10000])

def achievable_loss_for_size(m):
    # Vary |H| by coarsening the candidate-threshold grid to m allowed thresholds.
    grid = np.unique(np.linspace(1, E_MAX, m).astype(int))
    return float(loss_L(grid, e_all, y_all).min())

achievable = np.array([achievable_loss_for_size(int(m)) for m in H_SIZES])

def render_tradeoff(N=200, delta=0.05):
    # Hoeffding/union sample-complexity gap term: eps = sqrt(log(2|H|/delta) / (2N)).
    gap = np.sqrt(np.log(2 * H_SIZES / delta) / (2 * N))
    total = achievable + gap
    best = int(np.argmin(total))
    fig, ax = plt.subplots(figsize=(8, 4.5))
    ax.plot(H_SIZES, achievable, 'o-', color='#2ca02c', label='best achievable loss')
    ax.plot(H_SIZES, gap, 's-', color='#ff7f0e', label='generalization-gap bound term')
    ax.plot(H_SIZES, total, '^--', color='#d62728', label='sum (estimated risk)')
    ax.axvline(H_SIZES[best], color='black', ls=':', label=f'sweet spot |H| = {H_SIZES[best]}')
    ax.set_xscale('log'); ax.set_xlabel('model complexity |H| (log scale)'); ax.set_ylabel('loss')
    ax.set_title(f'Complexity tradeoff at N = {N}, delta = {delta}')
    ax.legend(); plt.tight_layout()
    render_tradeoff.fig = fig
    plt.show()
    return [list(row) for row in zip(H_SIZES.tolist(), achievable.round(3).tolist(), gap.round(3).tolist(), total.round(3).tolist())]

tradeoff_table = render_tradeoff(N=200, delta=0.05)
complexity_tradeoff_fig = render_tradeoff.fig
if HAS_WIDGETS:
    interact(render_tradeoff,
             N=IntSlider(value=200, min=50, max=2000, step=50, description='N'),
             delta=FloatSlider(value=0.05, min=0.01, max=0.30, step=0.01, description='delta'))

## Wrap-up: the whole arc in one breath

We went from a picture to a generalization guarantee:

1. **Feature** - edge-pixel count e turns an image into one number.
2. **Model** - the threshold rule f_h, with hypothesis set H and complexity |H| = 10000.
3. **Loss** - the 0-1 error L(h, D); the ideal threshold minimizes it on D_all.
4. **Gap** - optimizing on a finite D_train can look better than reality (the generalization gap).
5. **Bound** - union bound + Hoeffding give P(bad) <= |H| * 2 * exp(-2 * N * eps^2).
6. **Tradeoff** - larger |H| lowers achievable loss but widens the gap; the best |H| minimizes the sum.

This is the **three-step ML framework** in miniature - (1) a function with unknown parameters, (2) a loss, (3) optimization - wrapped in a generalization theory that says *why* step 3 on training data is trustworthy.

**Takeaway - Yes, Deep Learning:** deep models are attractive because they can drive the achievable loss far down; the bound reminds us to control complexity (more data, regularization, structure) so the gap stays small.

*Optional asides:* the single-hypothesis Hoeffding inequality is the per-h building block; rearranging the bound gives the **sample complexity** N >= log(2|H|/delta) / (2 * eps^2) needed for confidence 1 - delta.